# ConvLSTM Model for RTNet Task

## Task Description
- **Task**: RTNet (Reaction Time Network) - Human perceptual decision-making task
- **Stimuli**: MNIST digits (0-9, mapped to 8 classes for RTNet)
- **Raw Data Source**: RTNet behavioral dataset (human responses and RTs)
- **Goal**: Predict both classification and reaction time (RT) simultaneously

## Model Architecture
- ConvLSTM-based model with differentiable decision function (DiffDecision)
- Evidence accumulation with learnable threshold
- Noise injection capability (dropout + Gaussian noise)

Based on example.ipynb architecture with added noise injection capability.

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import os

from preprocess_mnist_behavioral import MNISTBehavioralDataset

%matplotlib inline

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    pass

In [ ]:
# Check device
if torch.cuda.is_available():
    device = torch.device('cuda:0')
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using Apple MPS")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"Device: {device}")

## 2. Define Model Components

In [ ]:
def add_noise(x, mask_p=0.0, std=0.0, rescale_after_dropout=True):
    if mask_p == 0 and std == 0:
        return x

    x_noisy = x.clone()

    if mask_p > 0:
        mask = torch.bernoulli(torch.ones_like(x) * (1 - mask_p))
        x_noisy = x_noisy * mask
        if rescale_after_dropout:
            x_noisy = x_noisy / (1 - mask_p + 1e-8)

    if std > 0:
        noise = torch.randn_like(x) * std
        x_noisy = x_noisy + noise

    return x_noisy

In [ ]:
class DiffDecision(torch.autograd.Function):
    @staticmethod
    def forward(ctx, trajectory, dsdt_trajectory):
        mask = trajectory > 0
        decision_time = mask.float().argmax(dim=1).float()
        decision_time[mask.sum(dim=1) == 0] = float(trajectory.shape[1] - 1)
        ctx.save_for_backward(dsdt_trajectory, decision_time, trajectory)
        return decision_time

    @staticmethod
    def backward(ctx, grad_output):
        dsdt_trajectory, decision_times, trajectory = ctx.saved_tensors
        device = dsdt_trajectory.device
        mask = trajectory > 0
        idx1 = (mask.sum(dim=1) == 0)
        idx2 = dsdt_trajectory[torch.arange(dsdt_trajectory.size(0), device=device), decision_times.long()] < 0
        idx = torch.logical_and(idx1, idx2)
        grads = torch.zeros_like(dsdt_trajectory)
        batch_indices = torch.arange(decision_times.size(0), device=device)
        grads[batch_indices, decision_times.long()] = -1.0 / (dsdt_trajectory[batch_indices, decision_times.long()] + 1e-6)
        grads[batch_indices[idx], decision_times[idx].long()] = 1e-6
        grads = grads * grad_output.unsqueeze(1)
        return grads, None

In [ ]:
class ConvLSTM(nn.Module):
    def __init__(self, input_channel, num_filter, kernel_size, stride=1, padding=1):
        super().__init__()
        self._conv = nn.Conv2d(
            in_channels=input_channel + num_filter,
            out_channels=num_filter * 4,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding
        )

        self.Wci = nn.Parameter(torch.zeros(1, num_filter, 1, 1))
        self.Wcf = nn.Parameter(torch.zeros(1, num_filter, 1, 1))
        self.Wco = nn.Parameter(torch.zeros(1, num_filter, 1, 1))

        self._input_channel = input_channel
        self._num_filter = num_filter

    def forward(self, inputs=None, states=None, seq_len=20):
        device = inputs.device
        B, _, H, W = inputs[0].shape

        if states is None:
            c = torch.zeros((B, self._num_filter, H, W), dtype=torch.float, device=device)
            h = torch.zeros_like(c)
        else:
            h, c = states

        outputs = []
        for t in range(seq_len):
            x = inputs[t]
            cat_x = torch.cat([x, h], dim=1)
            conv_x = self._conv(cat_x)

            i, f, new_c, o = torch.chunk(conv_x, 4, dim=1)
            i = torch.sigmoid(i + self.Wci * c)
            f = torch.sigmoid(f + self.Wcf * c)
            c = f * c + i * torch.tanh(new_c)
            o = torch.sigmoid(o + self.Wco * c)
            h = o * torch.tanh(c)

            outputs.append(h)

        outputs = torch.stack(outputs, dim=0)
        return outputs, (h, c)

In [ ]:
class RTify_ConvLSTM(nn.Module):
    def __init__(
        self,
        input_channel: int,
        num_filter: int,
        kernel_size: int,
        output_size: int,
        time_steps: int = 20,
        sigma: float = 2.0,
        noise_position: str = 'evidence',
        evidence_noise_std: float = 0.0,
        evidence_mask_p: float = 0.0,
        evidence_dropout_rescale: bool = False
    ):
        super(RTify_ConvLSTM, self).__init__()
        self.time_steps = time_steps
        self.noise_position = noise_position
        self.evidence_noise_std = evidence_noise_std
        self.evidence_mask_p = evidence_mask_p
        self.evidence_dropout_rescale = evidence_dropout_rescale
        
        self.convlstm = ConvLSTM(
            input_channel=input_channel,
            num_filter=num_filter,
            kernel_size=kernel_size
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(num_filter, output_size)

        self.evidence = nn.Sequential(
            nn.Linear(num_filter, num_filter),
            nn.ReLU(),
            nn.Linear(num_filter, 1),
            nn.Tanh()
        )

        self.threshold = torch.nn.Parameter(torch.tensor(6.0))
        self.sigma = sigma
        
    def forward(self, x):
        device = x.device
        B, C, H, W = x.shape

        x_seq = x.unsqueeze(0).repeat(self.time_steps, 1, 1, 1, 1)
        hidden_states, (h, c) = self.convlstm(x_seq, seq_len=self.time_steps)

        time_steps, B, num_filter, H, W = hidden_states.shape
        hidden_2d = hidden_states.view(time_steps * B, num_filter, H, W)
        pooled_2d = self.pool(hidden_2d).squeeze()
        hidden_states = pooled_2d.view(time_steps, B, num_filter)

        logit_trajectory = self.fc(hidden_states).squeeze().permute(1, 0, 2)
        
        s_traj = self.evidence(hidden_states).squeeze(-1).permute(1, 0)
        
        if self.noise_position in ['evidence', 'both']:
            s_traj = add_noise(
                s_traj,
                self.evidence_mask_p,
                self.evidence_noise_std,
                rescale_after_dropout=self.evidence_dropout_rescale
            )
        
        s_accumulated = torch.cumsum(s_traj, dim=1)
        dsdt_trajectory = torch.diff(s_accumulated, dim=1)
        dsdt_trajectory = torch.cat((dsdt_trajectory[:, 0].unsqueeze(1), dsdt_trajectory), dim=1)
        decision_time = DiffDecision.apply(s_accumulated - self.threshold, dsdt_trajectory)
        
        soft_index = torch.exp(-0.5 * (decision_time.unsqueeze(1) - torch.arange(self.time_steps, device=device)) ** 2 / self.sigma ** 2)
        soft_index = soft_index / soft_index.sum(dim=-1, keepdim=True)
        decision_logits = (logit_trajectory * soft_index.unsqueeze(-1)).sum(dim=1)
                
        return decision_logits, (decision_time+1) / self.time_steps

## 3. Load Data

In [ ]:
# Configuration
DATA_PATH = 'RTNet_Dataset/behavioral data.csv'
BATCH_SIZE = 64
TEST_SPLIT = 0.2
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
print("Loading dataset...")
full_dataset = MNISTBehavioralDataset(DATA_PATH, image_size=28)

total_len = len(full_dataset)
train_size = int((1 - TEST_SPLIT) * total_len)
test_size = total_len - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, test_size],
    generator=torch.Generator().manual_seed(RANDOM_SEED)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## 4. Create Model

In [ ]:
model = RTify_ConvLSTM(
    input_channel=1,
    num_filter=16,
    kernel_size=3,
    output_size=10,
    time_steps=20,
    sigma=2.0,
    noise_position='evidence',
    evidence_noise_std=0.5,
    evidence_mask_p=0.4,
    evidence_dropout_rescale=False
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Initial threshold: {model.threshold.item():.4f}")

## 5. Training

In [ ]:
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3
USE_RT_LOSS = True

label_criterion = nn.CrossEntropyLoss()
rt_criterion = nn.MSELoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

rt_loss, label_loss, acc, corr = [], [], [], []

In [ ]:
print("Starting Training...")
print(f"RT Supervision: {USE_RT_LOSS}")
print("="*60)

for epoch in range(NUM_EPOCHS):
    model.train()
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for batch in pbar:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        rt = batch['rt_normalized'].to(device)

        optimizer.zero_grad()
        decision_logits, rt_pred = model(images)

        rt_loss_temp = rt_criterion(rt_pred, rt)
        label_loss_temp = label_criterion(decision_logits, labels)

        if USE_RT_LOSS:
            total_loss = rt_loss_temp + label_loss_temp
        else:
            total_loss = label_loss_temp

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        rt_loss.append(rt_loss_temp.item())
        label_loss.append(label_loss_temp.item())
        acc.append((decision_logits.argmax(-1) == labels).float().mean().item())

        rt_pred_np = rt_pred.detach().cpu().numpy().flatten()
        rt_np = rt.cpu().numpy().flatten()
        corr_temp = np.corrcoef(rt_pred_np, rt_np)[0, 1] if len(rt_pred_np) > 1 else 0.0
        corr.append(np.nan_to_num(corr_temp))

        pbar.set_postfix({'loss': f'{total_loss.item():.4f}', 'acc': f'{acc[-1]:.3f}'})

print("\nTraining complete!")

## 6. Training Curves

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10, 5))
ax = ax.flatten()

ax[0].plot(rt_loss, '-k')
ax[0].set_title('RT loss')
ax[0].set_xlabel('iteration')
ax[0].set_ylabel('loss')

ax[1].plot(label_loss, '-k')
ax[1].set_title('Label loss')
ax[1].set_xlabel('iteration')
ax[1].set_ylabel('loss')

ax[2].plot(acc, '-k')
ax[2].set_title('Accuracy')
ax[2].set_xlabel('iteration')
ax[2].set_ylabel('accuracy')

ax[3].plot(corr, '-k')
ax[3].set_ylim(-1, 1)
ax[3].set_title('RT correlation')
ax[3].set_xlabel('iteration')
ax[3].set_ylabel('correlation')

plt.tight_layout()
plt.show()

## 7. Final Evaluation

In [ ]:
model.eval()

all_rt_pred = []
all_rt_human = []
all_labels = []
all_preds = []
all_correct = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        rt_human = batch['rt_normalized'].to(device)
        correct = batch['correct'].to(device)

        decision_logits, decision_time = model(images)
        pred_labels = decision_logits.argmax(dim=-1)

        all_rt_pred.extend(decision_time.cpu().numpy())
        all_rt_human.extend(rt_human.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(pred_labels.cpu().numpy())
        all_correct.extend(correct.cpu().numpy())

all_rt_pred = np.array(all_rt_pred)
all_rt_human = np.array(all_rt_human)
all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_correct = np.array(all_correct)

accuracy = np.mean(all_preds == all_labels)
correlation = np.corrcoef(all_rt_pred, all_rt_human)[0, 1]

correct_rt = all_rt_pred[all_correct == 1]
incorrect_rt = all_rt_pred[all_correct == 0]

print("\n" + "="*60)
print("Final Test Results")
print("="*60)
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"RT Correlation: {correlation:.4f}")
print(f"Learned Threshold: {model.threshold.item():.4f}")
print(f"\nRT by Correctness (normalized):")
print(f"  Correct trials: {correct_rt.mean():.4f} +/- {correct_rt.std():.4f} (n={len(correct_rt)})")
if len(incorrect_rt) > 0:
    print(f"  Incorrect trials: {incorrect_rt.mean():.4f} +/- {incorrect_rt.std():.4f} (n={len(incorrect_rt)})")

## 8. RT Distribution Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(all_rt_human, all_rt_pred, alpha=0.3, s=10)
axes[0].set_xlabel('Human RT (normalized)')
axes[0].set_ylabel('Model RT (normalized)')
axes[0].set_title(f'Model vs Human RT (r={correlation:.3f})')
axes[0].grid(True, alpha=0.3)

z = np.polyfit(all_rt_human, all_rt_pred, 1)
p = np.poly1d(z)
axes[0].plot(all_rt_human, p(all_rt_human), "r--", alpha=0.8, label='fit line')
axes[0].legend()

axes[1].hist(correct_rt, bins=30, alpha=0.6, label=f'Correct (n={len(correct_rt)})', density=True)
if len(incorrect_rt) > 0:
    axes[1].hist(incorrect_rt, bins=30, alpha=0.6, label=f'Incorrect (n={len(incorrect_rt)})', density=True)
axes[1].set_xlabel('RT (normalized)')
axes[1].set_ylabel('Density')
axes[1].set_title('RT Distribution by Correctness')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Save Model

In [ ]:
OUTPUT_DIR = './output_mnist_convlstm'
os.makedirs(OUTPUT_DIR, exist_ok=True)

model_path = os.path.join(OUTPUT_DIR, 'convlstm_mnist.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'final_accuracy': accuracy,
    'final_correlation': correlation,
    'final_threshold': model.threshold.item()
}, model_path)

print(f"Model saved to: {model_path}")

results_df = pd.DataFrame({
    'true_label': all_labels,
    'pred_label': all_preds,
    'correct': all_correct,
    'rt_pred_normalized': all_rt_pred,
    'rt_human_normalized': all_rt_human,
    'rt_pred_seconds': full_dataset.denormalize_rt(all_rt_pred),
    'rt_human_seconds': full_dataset.denormalize_rt(all_rt_human)
})

results_path = os.path.join(OUTPUT_DIR, 'results.csv')
results_df.to_csv(results_path, index=False)
print(f"Results saved to: {results_path}")